# Mega Project 2 — Regulatory Capital & Expected Loss
## Problem 3: Economic Capital & Unexpected Loss — Real Monte Carlo Simulation
## of Notebook 01's PD/LGD/EAD/Correlation Inputs

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
Notebook 01 computes a real Expected Loss and a Basel-retail-IRB capital
requirement via a **closed-form** solution of the single-factor Vasicek/ASRF
model — a single number, the 99.9th percentile point estimate. This notebook
asks the question a risk function needs answered next: what does the **full**
loss distribution look like, what is Value-at-Risk and Expected Shortfall at
confidence levels *other* than 99.9%, and does an independent numerical
simulation of the same model agree with the closed form?

### This notebook trains no model and introduces no new assumption
It reuses Notebook 01's already-computed, already-disclosed real per-applicant
PD, LGD, EAD, and asset correlation (hard dependency — fails loudly if
Notebook 01 has not been run yet). No new LGD/correlation/PD value is
introduced anywhere in this notebook.

### What is genuinely new here
A real, vectorized Monte Carlo simulation of the systematic risk factor in
the same single-factor Vasicek model Notebook 01's closed form already
solves analytically — giving real Value-at-Risk / Expected Shortfall /
Economic Capital at **multiple** documented confidence levels (95%, 99%,
99.5%, 99.9%), plus a real independent cross-check of Notebook 01's
closed-form capital number.

### Limitation, stated plainly
This simulates the SAME single-factor, infinite-granularity (ASRF)
assumptions Basel's own closed form already makes — it is a genuine
cross-check and a genuine source of new distributional detail, not a more
sophisticated economic-capital model (no granularity adjustment, no
multi-factor correlation, no single-name concentration add-on).

### Technique (vectorized Monte Carlo, not per-applicant or per-resample loops)
Every systematic-factor draw's conditional default probability is computed
for the WHOLE real portfolio in one vectorized `scipy.stats.norm.cdf` call;
draws are processed in batches so the number of Python-level loop iterations
stays small regardless of how many total draws are requested — the same
vectorize-the-inner-loop discipline as this suite's bootstrap resampling.

### Lessons applied from Mega Project 1's hardening history (built in from
### this notebook's first version)
- WARP hardware fix before any heavy import.
- Two-tier "Pipeline Integrity" vs. "Statistical Robustness" verdict
  separation — here, robustness means real independent-reseed Monte Carlo
  convergence plus the closed-form cross-check (no real TARGET to test a
  classifier against in this notebook).
- HYPER reuse: `src/reporting/report_builder.py` — nothing redefined locally.

### What this notebook does
1. Loads Notebook 01's real per-applicant PD/LGD/EAD/correlation output
   (hard dependency).
2. Runs a real, vectorized, batched Monte Carlo simulation of the
   single-factor Vasicek model (documented, user-overridable draw count via
   `project_config.json` `"mc_draws"`).
3. Computes real VaR / Expected Shortfall / Economic Capital at 4 documented
   confidence levels directly from the simulated loss distribution.
4. Cross-checks the 99.9% Economic Capital against Notebook 01's real
   closed-form Basel capital requirement (documented tolerance).
5. Runs a real independent-reseed convergence check (this notebook's
   statistical-robustness family).
6. Reports a two-tier verdict, then generates the full Stage-5 reporting
   package (CSV, Word, Excel with an Assumptions sheet, HTML dashboard).

### Verification status
Verified end-to-end on the synthetic fixture via real Jupyter execution — 0
errors, all integrity checks pass, HTML dashboard confirmed under a
network-blocked Playwright check, Excel workbook confirmed via LibreOffice
headless recalculation. **Not yet run against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 03 — MEGA PROJECT 2: REGULATORY CAPITAL & EXPECTED LOSS
# PROBLEM 3: ECONOMIC CAPITAL & UNEXPECTED LOSS
# Real per-applicant PD/LGD/EAD/correlation (Notebook 01, reused not
# recomputed), propagated through a real vectorized Monte Carlo simulation
# of the same single-factor Vasicek/ASRF model underlying Notebook 01's
# closed-form Basel capital charge -- to obtain a real, simulated portfolio
# loss DISTRIBUTION (not just its 99.9th-percentile closed-form point),
# real Value-at-Risk and Expected Shortfall at multiple documented
# confidence levels, and Economic Capital (VaR - Expected Loss) at each.
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains no model and introduces
# NO new PD/LGD/EAD/correlation assumption -- it reuses Notebook 01's
# already-computed, already-disclosed real per-applicant PD, LGD_ASSUMED,
# EAD_PROXY, and CORRELATION_R (decision_engine/artifacts/
# notebook_01_capital_scores.csv), unchanged. Hard dependency: fails loudly
# and immediately if Notebook 01 has not been run yet.
#
# WHAT IS GENUINELY NEW HERE (vs. Notebook 01): Notebook 01's Basel capital
# requirement is a CLOSED-FORM point estimate -- the analytic 99.9th
# percentile of the single-factor Vasicek loss distribution, solved directly
# via the K() formula, with no visibility into the rest of the distribution.
# This notebook instead SIMULATES that same distribution directly (same
# model, same PD/LGD/EAD/R inputs, zero new assumptions) via a real Monte
# Carlo over the systematic risk factor, which is the only way to obtain:
#   - Value-at-Risk / Economic Capital at confidence levels OTHER than 99.9%
#     (95%, 99%, 99.5% -- useful for internal risk appetite, not just the
#     Pillar 1 minimum).
#   - Expected Shortfall / CVaR (mean loss beyond VaR) -- has no closed form
#     under this model at all.
#   - A real, independent NUMERICAL CROSS-CHECK of Notebook 01's closed-form
#     capital number: if the Monte-Carlo-simulated 99.9% Economic Capital and
#     the closed-form Basel capital requirement disagree by more than a
#     documented tolerance, that is flagged, not hidden -- see Section 8.
#
# LIMITATION, STATED PLAINLY (read before treating this as a full internal
# economic-capital model): this Monte Carlo simulates the SAME single-factor,
# infinite-granularity (ASRF) assumptions Basel's own closed form already
# makes -- one systematic factor, no single-name concentration/granularity
# add-on, no multi-factor structure. It is a genuine, independently-computed
# cross-check and a genuine source of new distributional detail (VaR at
# other levels, Expected Shortfall), not a more sophisticated model than
# Notebook 01 -- it would not, by construction, ever produce a materially
# different 99.9% figure than the closed form on a well-diversified retail
# portfolio, and a real bank's internal economic-capital model would go
# further (granularity adjustment, multi-factor correlation, name
# concentration) than this notebook's scope.
#
# TECHNIQUE (vectorized, not a per-applicant or per-resample loop): this is a
# real Monte Carlo simulation of a parametric model, not a bootstrap
# resampling of empirical data -- a different technique from this suite's
# "vectorized multinomial bootstrap" lesson (which resamples an empirical
# joint distribution). Per systematic-factor draw, every real applicant's
# conditional default probability is computed via ONE vectorized
# `scipy.stats.norm.cdf` call over the whole real portfolio at once (never a
# per-applicant Python loop), and draws are processed in BATCHES (a
# (batch_size, n_applicants) matrix per batch, one vectorized call each) so
# the number of Python-level loop iterations is small regardless of how many
# total draws are requested -- the same vectorize-the-inner-loop discipline
# already applied to this suite's bootstrap resampling, WARP hardware
# governance, and RAM-headroom checks throughout the batch loop.
#
# LESSONS APPLIED FROM MEGA PROJECT 1'S HARDENING HISTORY (built in from this
# notebook's first version): WARP hardware fix before any heavy import;
# two-tier "Pipeline Integrity" vs. "Statistical Robustness" verdict
# separation (here: a real independent-reseed convergence check plus the
# closed-form cross-check stand in for the classifier-vs-TARGET tests used
# elsewhere, since this notebook validates a simulation, not a classifier);
# HYPER shared-module reuse (report_builder, regulatory_capital_features);
# never git operations via the device-mounted folder.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

SEED = int(CONFIG.get("random_seed", 42))
# Monte Carlo draw count is a documented, user-overridable performance knob
# (project_config.json "mc_draws"), NOT hardcoded to whatever happens to run
# fast in this suite's cloud-fixture environment. Default chosen so the 99%
# VaR/ES estimate has a reasonably large effective tail sample (~500 draws
# at/above the 99th percentile) while staying fast (see Section 6 -- the
# whole loop is a small number of vectorized batch calls, not one call per
# draw or per applicant).
N_MC_DRAWS = int(CONFIG.get("mc_draws", 50_000))
N_MC_DRAWS_CHECK = int(CONFIG.get("mc_draws_reseed_check", 20_000))
MC_BATCH_SIZE = int(CONFIG.get("mc_batch_size", 500))
CONFIDENCE_LEVELS = [0.95, 0.99, 0.995, 0.999]

MP2_ARTIFACTS_DIR = SUITE_ROOT / "02_mega_project_2_regulatory_capital" / "decision_engine" / "artifacts"
ARTIFACTS_DIR = MP2_ARTIFACTS_DIR
REPORTS_DIR = SUITE_ROOT / "02_mega_project_2_regulatory_capital" / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity, check_ram_headroom

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import)
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import norm

T0 = time.time()

from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}; N_MC_DRAWS = {N_MC_DRAWS:,} "
      f"(+ {N_MC_DRAWS_CHECK:,} independent-reseed draws for the convergence check), "
      f"batch size = {MC_BATCH_SIZE}")

# ---------------------------------------------------------------------------
# SECTION 4 — Load Notebook 01's real per-applicant output (HARD dependency)
# ---------------------------------------------------------------------------
NB01_SCORES_PATH = ARTIFACTS_DIR / "notebook_01_capital_scores.csv"
if not NB01_SCORES_PATH.exists():
    raise FileNotFoundError(
        "Mega Project 2 / Notebook 03 requires Mega Project 2 / Notebook 01's real "
        "per-applicant output, which has not been produced on this machine yet. Fix: run "
        "02_mega_project_2_regulatory_capital/notebooks/01_expected_loss_capital_requirement.ipynb "
        "end-to-end first, then re-run this notebook."
    )
scores = pl.read_csv(NB01_SCORES_PATH)
N_SCOPE = scores.height
print(f"[LOAD] Real per-applicant PD/LGD/EAD/correlation output from Notebook 01: {N_SCOPE:,} rows "
      f"(reused unchanged -- no new PD/LGD/EAD/correlation assumption introduced here).")

required_cols = ["PD", "LGD_ASSUMED", "EAD_PROXY", "CORRELATION_R", "EXPECTED_LOSS",
                  "CAPITAL_REQUIREMENT", "CAPITAL_SEGMENT"]
missing_cols = [c for c in required_cols if c not in scores.columns]
if missing_cols:
    raise KeyError(
        f"Required columns missing from notebook_01_capital_scores.csv: {missing_cols}. "
        "Re-run Notebook 01 (it may be an older output format)."
    )

PD_ARR = np.clip(scores["PD"].to_numpy().astype(np.float64), 1e-6, 1 - 1e-6)
LGD_ARR = scores["LGD_ASSUMED"].to_numpy().astype(np.float64)
EAD_ARR = scores["EAD_PROXY"].to_numpy().astype(np.float64)
R_ARR = np.clip(scores["CORRELATION_R"].to_numpy().astype(np.float64), 1e-6, 1 - 1e-6)
LGD_EAD_ARR = LGD_ARR * EAD_ARR

REAL_TOTAL_EL = float(scores["EXPECTED_LOSS"].to_numpy().sum())
REAL_TOTAL_EAD = float(EAD_ARR.sum())
CLOSED_FORM_TOTAL_CAPITAL = float(scores["CAPITAL_REQUIREMENT"].to_numpy().sum())
check_ram_headroom(PERF)

# ---------------------------------------------------------------------------
# SECTION 5 — Precompute per-applicant Vasicek single-factor constants.
# Conditional default probability given systematic factor Z (the same
# formula underlying Notebook 01's closed-form K(), [BCBS05]):
#   p_i(Z) = Phi[ (Phi^-1(PD_i) - sqrt(R_i) * Z) / sqrt(1 - R_i) ]
# Precomputing a_i = Phi^-1(PD_i)/sqrt(1-R_i) and b_i = sqrt(R_i/(1-R_i))
# ONCE (they do not depend on Z) turns every draw's inner term into a single
# vectorized multiply-subtract, not a repeated call to norm.ppf per draw.
# ---------------------------------------------------------------------------
PD_PPF_ARR = norm.ppf(PD_ARR)
A_CONST = PD_PPF_ARR / np.sqrt(1.0 - R_ARR)
B_CONST = np.sqrt(R_ARR / (1.0 - R_ARR))
print(f"[MODEL] Real per-applicant Vasicek single-factor constants precomputed for all "
      f"{N_SCOPE:,} real applicants (same PD/LGD/EAD/R as Notebook 01 -- [BCBS05] single-factor "
      f"ASRF derivation, unmodified).")


def _simulate_portfolio_losses(rng: np.random.Generator, n_draws: int, batch_size: int) -> np.ndarray:
    """Real vectorized Monte Carlo of the single-factor Vasicek portfolio loss
    distribution. Draws n_draws systematic-factor realizations Z ~ N(0,1),
    processed in batches (one vectorized (batch, n_applicants) norm.cdf call
    per batch, not one call per draw or per applicant), and returns the
    n_draws real simulated total-portfolio-loss values -- the infinite-
    granularity (ASRF) approximation, i.e. within-batch idiosyncratic risk is
    assumed diversified away exactly as Basel's own closed form assumes (see
    module-level LIMITATION note above), so the conditional loss given Z is
    the deterministic sum of conditional expected losses, not a further
    per-applicant Bernoulli simulation.
    """
    losses = np.empty(n_draws, dtype=np.float64)
    n_done = 0
    n_batches_done = 0
    while n_done < n_draws:
        this_batch = min(batch_size, n_draws - n_done)
        z = rng.standard_normal(this_batch)
        # inner[m, i] = A_CONST[i] - B_CONST[i] * z[m]  -- (this_batch, N) matrix
        inner = A_CONST[None, :] - B_CONST[None, :] * z[:, None]
        cond_pd = norm.cdf(inner)                      # vectorized, one call for the whole batch
        losses[n_done:n_done + this_batch] = cond_pd @ LGD_EAD_ARR   # (this_batch,) real simulated losses
        n_done += this_batch
        n_batches_done += 1
        if n_batches_done % 10 == 0:
            check_ram_headroom(PERF)
    return losses


# ---------------------------------------------------------------------------
# SECTION 6 — Main Monte Carlo run (real, vectorized, batched).
# ---------------------------------------------------------------------------
_t_mc = time.time()
rng_main = np.random.default_rng(SEED)
SIMULATED_LOSSES = _simulate_portfolio_losses(rng_main, N_MC_DRAWS, MC_BATCH_SIZE)
MC_RUNTIME_S = round(time.time() - _t_mc, 1)
print(f"[MONTE CARLO] {N_MC_DRAWS:,} real vectorized systematic-factor draws simulated in "
      f"{MC_RUNTIME_S}s (batch size {MC_BATCH_SIZE}). Simulated mean portfolio loss: "
      f"${SIMULATED_LOSSES.mean():,.0f} (real closed-form Expected Loss from Notebook 01: "
      f"${REAL_TOTAL_EL:,.0f} -- these should closely agree by construction of the same model; "
      f"see Section 8).")

# ---------------------------------------------------------------------------
# SECTION 7 — Real VaR / Expected Shortfall / Economic Capital at each
# documented confidence level, computed directly from the simulated
# distribution (no closed form used here).
# ---------------------------------------------------------------------------
def _var_es_ec(losses: np.ndarray, alpha: float, el: float) -> dict:
    """VaR/ES/EC directly from the empirical simulated distribution -- no
    closed-form or analytic-approximation standard error is reported here
    (an analytic quantile-SE formula would need a density estimate at the
    quantile, which is a real but separate derivation this notebook does not
    attempt); `n_tail_draws` is reported instead as an honest, real
    diagnostic of how few simulated draws inform the estimate at extreme
    alpha, and Section 9's independent-reseed convergence check is this
    notebook's actual empirical evidence of estimate stability."""
    var = float(np.quantile(losses, alpha))
    tail = losses[losses >= var]
    es = float(tail.mean()) if len(tail) > 0 else var
    ec = var - el
    return {
        "confidence_level": alpha,
        "value_at_risk": var,
        "expected_shortfall": es,
        "economic_capital": ec,
        "n_tail_draws": int(len(tail)),
    }


risk_metrics = [_var_es_ec(SIMULATED_LOSSES, a, REAL_TOTAL_EL) for a in CONFIDENCE_LEVELS]
risk_metrics_df = pd.DataFrame(risk_metrics)
_var_seq = risk_metrics_df["value_at_risk"].tolist()
VAR_NONDECREASING = all(_var_seq[i] <= _var_seq[i + 1] + 1e-6 for i in range(len(_var_seq) - 1))
for m in risk_metrics:
    print(f"[RISK] alpha={m['confidence_level']:.1%}: real simulated VaR=${m['value_at_risk']:,.0f}, "
          f"Expected Shortfall=${m['expected_shortfall']:,.0f}, Economic Capital=${m['economic_capital']:,.0f} "
          f"(n_tail_draws={m['n_tail_draws']:,}).")

EC_999 = float(risk_metrics_df.loc[risk_metrics_df["confidence_level"] == 0.999, "economic_capital"].iloc[0])
VAR_999 = float(risk_metrics_df.loc[risk_metrics_df["confidence_level"] == 0.999, "value_at_risk"].iloc[0])

# ---------------------------------------------------------------------------
# SECTION 8 — Real cross-check vs. Notebook 01's closed-form Basel capital
# requirement. Both numbers come from the SAME single-factor Vasicek model
# and the SAME real PD/LGD/EAD/R inputs -- one solved analytically (Notebook
# 01's K()), one solved by real Monte Carlo simulation here. They should
# agree closely; a documented, disclosed tolerance (not tuned per-run)
# accounts for real Monte Carlo sampling error at an extreme (99.9%)
# quantile with a finite number of draws.
# ---------------------------------------------------------------------------
CLOSED_FORM_VS_MC_REL_DIFF = (
    abs(EC_999 - CLOSED_FORM_TOTAL_CAPITAL) / CLOSED_FORM_TOTAL_CAPITAL
    if CLOSED_FORM_TOTAL_CAPITAL > 0 else float("nan")
)
CROSS_CHECK_TOLERANCE = 0.10  # documented: 10% relative tolerance at the 99.9% quantile
CROSS_CHECK_PASSES = CLOSED_FORM_VS_MC_REL_DIFF <= CROSS_CHECK_TOLERANCE
print(f"[CROSS-CHECK] Real closed-form Basel capital requirement (Notebook 01): "
      f"${CLOSED_FORM_TOTAL_CAPITAL:,.0f}. Real Monte-Carlo-simulated 99.9% Economic Capital "
      f"(this notebook): ${EC_999:,.0f}. Relative difference: {CLOSED_FORM_VS_MC_REL_DIFF:.2%} "
      f"(documented tolerance: {CROSS_CHECK_TOLERANCE:.0%}) -> "
      f"{'WITHIN TOLERANCE' if CROSS_CHECK_PASSES else 'EXCEEDS TOLERANCE'}.")

# ---------------------------------------------------------------------------
# SECTION 9 — Real independent-reseed convergence check (this notebook's
# "statistical robustness" family -- there is no real TARGET to test a
# classifier against here, so robustness instead means: does a SECOND,
# independently-seeded Monte Carlo run of the SAME model agree with the
# first, within a documented per-confidence-level tolerance? Analogous in
# spirit to this suite's split-half PSI checks elsewhere.). Tolerances are
# wider at higher confidence levels because a finite Monte Carlo sample has
# real, larger sampling error further into the tail -- a real, disclosed
# property of Monte Carlo estimation, not a threshold tuned to force a pass.
# ---------------------------------------------------------------------------
_t_check = time.time()
rng_check = np.random.default_rng(SEED + 1)
SIMULATED_LOSSES_CHECK = _simulate_portfolio_losses(rng_check, N_MC_DRAWS_CHECK, MC_BATCH_SIZE)
CHECK_RUNTIME_S = round(time.time() - _t_check, 1)
CHECK_TOLERANCE_BY_LEVEL = {0.95: 0.05, 0.99: 0.08, 0.995: 0.10, 0.999: 0.15}
reseed_metrics = [_var_es_ec(SIMULATED_LOSSES_CHECK, a, REAL_TOTAL_EL) for a in CONFIDENCE_LEVELS]
convergence_detail = []
convergence_holds = True
for m_main, m_check in zip(risk_metrics, reseed_metrics):
    alpha = m_main["confidence_level"]
    tol = CHECK_TOLERANCE_BY_LEVEL[alpha]
    rel_diff = (
        abs(m_main["economic_capital"] - m_check["economic_capital"]) / max(m_main["economic_capital"], 1.0)
    )
    ok = rel_diff <= tol
    convergence_holds = convergence_holds and ok
    convergence_detail.append({
        "confidence_level": alpha, "main_run_ec": m_main["economic_capital"],
        "reseed_run_ec": m_check["economic_capital"], "relative_difference": rel_diff,
        "documented_tolerance": tol, "converged": bool(ok),
    })
    print(f"[CONVERGENCE] alpha={alpha:.1%}: main-run EC=${m_main['economic_capital']:,.0f}, "
          f"independent-reseed EC=${m_check['economic_capital']:,.0f}, relative diff={rel_diff:.2%} "
          f"(tolerance {tol:.0%}) -> {'CONVERGED' if ok else 'NOT CONVERGED'}.")

# ---------------------------------------------------------------------------
# SECTION 10 — STATISTICAL ROBUSTNESS VERDICT (separate from Section 11's
# structural Pipeline Integrity Checks -- LESSON #2, applied to a simulation
# notebook's own equivalent of that pattern).
# ---------------------------------------------------------------------------
validation_checks = [
    ("monte_carlo_converged_across_independent_reseed", convergence_holds),
    ("closed_form_cross_check_within_tolerance", CROSS_CHECK_PASSES),
]
ANALYSIS_ROBUST = all(ok for _, ok in validation_checks)
_failed_validation_checks = [name for name, ok in validation_checks if not ok]
ANALYSIS_VERDICT = (
    "STATISTICALLY ROBUST — RECOMMENDED FOR PRODUCTION" if ANALYSIS_ROBUST
    else "NOT YET STATISTICALLY ROBUST — failed: " + ", ".join(_failed_validation_checks) +
         " (a separate, stricter quantitative-robustness gate, distinct from the structural "
         "pipeline integrity checks reported elsewhere in this notebook's output -- consider "
         "increasing N_MC_DRAWS via project_config.json \"mc_draws\" if this persists)"
)
for name, ok in validation_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Statistical robustness verdict: {ANALYSIS_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 11 — Inline charts
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(SIMULATED_LOSSES, bins=80, color=_palette(1)[0], alpha=0.85)
axes[0].axvline(REAL_TOTAL_EL, color="black", linestyle="--", linewidth=1.5, label="Expected Loss")
axes[0].axvline(VAR_999, color="crimson", linestyle="--", linewidth=1.5, label="VaR 99.9%")
axes[0].set_xlabel("Real Simulated Portfolio Loss ($)"); axes[0].set_ylabel("Draws")
axes[0].set_title("Real Simulated Portfolio Loss Distribution"); axes[0].legend()
_ec_colors = _palette(len(risk_metrics_df))
axes[1].bar([f"{c:.1%}" for c in risk_metrics_df["confidence_level"]],
            risk_metrics_df["economic_capital"], color=_ec_colors)
axes[1].axhline(CLOSED_FORM_TOTAL_CAPITAL, color="black", linestyle="--", linewidth=1.5,
                 label="Closed-form Basel capital (Notebook 01)")
axes[1].set_ylabel("Real Economic Capital ($)"); axes[1].set_title("Economic Capital by Confidence Level")
axes[1].legend()
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_03_economic_capital.png", dpi=110)
plt.show()

# ---------------------------------------------------------------------------
# SECTION 12 — Pipeline Integrity Checks (structural)
# ---------------------------------------------------------------------------
checks = [
    ("real_data_loaded", N_SCOPE > 0),
    ("required_columns_present", len(missing_cols) == 0),
    ("simulated_losses_finite_and_nonnegative", bool(np.isfinite(SIMULATED_LOSSES).all()
                                                        and (SIMULATED_LOSSES >= 0).all())),
    ("var_nondecreasing_by_confidence_level", VAR_NONDECREASING),
    ("economic_capital_nonnegative_at_99_and_above", bool(
        (risk_metrics_df.loc[risk_metrics_df["confidence_level"] >= 0.99, "economic_capital"] >= 0).all()
    )),
    ("reseed_run_completed", len(SIMULATED_LOSSES_CHECK) == N_MC_DRAWS_CHECK),
    ("cpu_thread_ceiling_applied_before_import", os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
]
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Pipeline integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 13 — Reporting & Packaging (SOP Stage 5)
# ---------------------------------------------------------------------------
csv_paths = write_csv_outputs(
    {"notebook_03_risk_metrics_by_confidence_level": risk_metrics_df},
    REPORTS_DIR,
)

ASSUMPTIONS = {
    "Simulation model": "Single-factor Vasicek/ASRF (same model as Notebook 01's closed-form K())",
    "Monte Carlo draws (main run)": f"{N_MC_DRAWS:,}",
    "Cross-check tolerance vs. closed form (99.9%)": f"{CROSS_CHECK_TOLERANCE:.0%}",
}
ASSUMPTION_NOTES = {
    "Simulation model": "[BCBS05] the same Vasicek/ASRF single-systematic-factor derivation underlying "
                        "Basel's own closed-form retail-IRB capital function; see also Vasicek, O. (2002), "
                        "\"The Distribution of Loan Portfolio Value,\" Risk, for the original single-factor "
                        "portfolio credit loss model. No new LGD/PD/correlation assumption is introduced -- "
                        "every input is Notebook 01's real, already-disclosed per-applicant value, unchanged.",
    "Monte Carlo draws (main run)": "A real, disclosed, user-overridable performance knob "
                                     "(project_config.json \"mc_draws\") -- not tuned per-run.",
    "Cross-check tolerance vs. closed form (99.9%)": "A documented allowance for real Monte Carlo sampling "
                                                       "error at an extreme quantile with a finite number of "
                                                       "draws -- not a threshold chosen to force a pass.",
}

STORY_LOSS_DIST = [
    f"Real simulated mean portfolio loss (${SIMULATED_LOSSES.mean():,.0f}) closely tracks Notebook 01's "
    f"real closed-form Expected Loss (${REAL_TOTAL_EL:,.0f}) -- expected, since both come from the same "
    f"real PD x LGD x EAD inputs; this is a real internal-consistency check, not a coincidence.",
    f"Real simulated 99.9% VaR: ${VAR_999:,.0f}. Real 99.9% Economic Capital (VaR - EL): ${EC_999:,.0f}, "
    f"vs. Notebook 01's real closed-form Basel capital requirement of ${CLOSED_FORM_TOTAL_CAPITAL:,.0f} "
    f"({CLOSED_FORM_VS_MC_REL_DIFF:.2%} relative difference).",
]
STORY_EC_CHART = [
    f"Real Economic Capital rises from ${risk_metrics[0]['economic_capital']:,.0f} at "
    f"{risk_metrics[0]['confidence_level']:.1%} confidence to ${EC_999:,.0f} at 99.9% -- by construction, "
    f"a real, monotonically non-decreasing quantile of the same simulated loss distribution.",
    f"Statistical robustness verdict: {ANALYSIS_VERDICT}.",
]
INSIGHTS = [{
    "headline": f"Real Monte Carlo economic-capital estimate {'confirms' if CROSS_CHECK_PASSES else 'diverges from'} "
                f"Notebook 01's closed-form Basel capital requirement",
    "specific": f"99.9% Economic Capital (Monte Carlo): ${EC_999:,.0f} vs. closed-form capital requirement: "
                f"${CLOSED_FORM_TOTAL_CAPITAL:,.0f} ({CLOSED_FORM_VS_MC_REL_DIFF:.2%} relative difference, "
                f"tolerance {CROSS_CHECK_TOLERANCE:.0%}).",
    "measurable": f"Real VaR/Expected Shortfall/Economic Capital computed at {len(CONFIDENCE_LEVELS)} "
                  f"documented confidence levels from {N_MC_DRAWS:,} real simulated draws.",
    "achievable": "No further tuning required this cycle." if ANALYSIS_ROBUST else
                  "Increase N_MC_DRAWS (project_config.json \"mc_draws\") to tighten the tail-quantile "
                  "estimate; re-run after any Notebook 01 update.",
    "relevant": "Gives risk management a real loss DISTRIBUTION (VaR at multiple levels, Expected "
                "Shortfall) that Notebook 01's single closed-form number cannot provide, plus an "
                "independent numerical check on that closed-form number.",
    "timebound": "Re-run after any Notebook 01 (MP2) re-run or LGD/correlation assumption update.",
}]

word_path = build_word_report(
    REPORTS_DIR / "notebook_03_report.docx",
    title="Mega Project 2 — Notebook 03: Economic Capital & Unexpected Loss",
    subtitle="Real Monte Carlo simulation of Notebook 01's PD/LGD/EAD/correlation inputs",
    exec_summary=[
        f"{N_SCOPE:,} real applicants (Notebook 01's real PD/LGD/EAD/correlation output, reused unchanged).",
        f"{N_MC_DRAWS:,} real vectorized Monte Carlo draws of the single-factor Vasicek model "
        f"({MC_RUNTIME_S}s runtime).",
        f"Real 99.9% Economic Capital: ${EC_999:,.0f} (closed-form cross-check: "
        f"{CLOSED_FORM_VS_MC_REL_DIFF:.2%} relative difference).",
        f"Statistical robustness verdict: {ANALYSIS_VERDICT}",
    ],
    sections=[
        {"heading": "Value-at-Risk, Expected Shortfall, and Economic Capital by Confidence Level",
         "paragraphs": ["Real figures computed directly from the simulated portfolio loss distribution "
                        "(not the closed form) at each documented confidence level."],
         "table": {"headers": ["Confidence Level", "VaR", "Expected Shortfall", "Economic Capital", "N Tail Draws"],
                   "rows": [[f"{m['confidence_level']:.1%}", f"${m['value_at_risk']:,.0f}",
                             f"${m['expected_shortfall']:,.0f}", f"${m['economic_capital']:,.0f}",
                             f"{m['n_tail_draws']:,}"] for m in risk_metrics]},
         "image_path": ARTIFACTS_DIR / "notebook_03_economic_capital.png", "story": STORY_LOSS_DIST + STORY_EC_CHART},
        {"heading": "Independent-Reseed Convergence Check",
         "paragraphs": ["A second, independently-seeded Monte Carlo run, compared against the main run "
                        "at each confidence level (this notebook's statistical-robustness family)."],
         "table": {"headers": ["Confidence Level", "Main Run EC", "Reseed Run EC", "Relative Diff", "Tolerance", "Converged"],
                   "rows": [[f"{d['confidence_level']:.1%}", f"${d['main_run_ec']:,.0f}",
                             f"${d['reseed_run_ec']:,.0f}", f"{d['relative_difference']:.2%}",
                             f"{d['documented_tolerance']:.0%}", "Yes" if d["converged"] else "No"]
                            for d in convergence_detail]},
         "story": [f"Statistical robustness verdict: {ANALYSIS_VERDICT}."]},
    ],
    insights=INSIGHTS,
)

excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_03_workbook.xlsx",
    assumptions=ASSUMPTIONS, assumption_notes=ASSUMPTION_NOTES,
    data_sheets=[
        {"name": "Risk Metrics", "headers": list(risk_metrics_df.columns),
         "rows": risk_metrics_df.astype(object).values.tolist(), "highlight_col": "economic_capital"},
        {"name": "Convergence Check", "headers": list(convergence_detail[0].keys()),
         "rows": [list(d.values()) for d in convergence_detail], "highlight_col": "converged"},
    ],
    formula_sheet={"name": "Portfolio Summary",
                   "rows": [("Real Total EAD", REAL_TOTAL_EAD), ("Real Total Expected Loss", REAL_TOTAL_EL),
                            ("Closed-form Basel Capital (Notebook 01)", CLOSED_FORM_TOTAL_CAPITAL),
                            ("Monte Carlo 99.9% Economic Capital", EC_999),
                            ("Closed-form vs. Monte Carlo relative difference", CLOSED_FORM_VS_MC_REL_DIFF)]},
    insights_sheet={"name": "SMART Insights", "items": INSIGHTS},
)

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_03_dashboard.html",
    title="Mega Project 2 — Economic Capital & Unexpected Loss",
    subtitle=f"{N_SCOPE:,} real applicants — {N_MC_DRAWS:,} real Monte Carlo draws",
    kpi_cards=[
        {"label": "Real Applicants", "value": f"{N_SCOPE:,}"},
        {"label": "Real Expected Loss", "value": f"${REAL_TOTAL_EL:,.0f}"},
        {"label": "99.9% Economic Capital", "value": f"${EC_999:,.0f}"},
        {"label": "vs. Closed-Form Capital", "value": f"{CLOSED_FORM_VS_MC_REL_DIFF:.2%} diff"},
    ],
    charts=[
        {"id": "ecByLevel", "title": "Economic Capital by Confidence Level", "type": "bar",
         "labels": [f"{c:.1%}" for c in risk_metrics_df["confidence_level"]],
         "datasets": [{"label": "Economic Capital ($)", "data": risk_metrics_df["economic_capital"].round(0).tolist()}],
         "story": STORY_EC_CHART},
        {"id": "esByLevel", "title": "Expected Shortfall by Confidence Level", "type": "bar",
         "labels": [f"{c:.1%}" for c in risk_metrics_df["confidence_level"]],
         "datasets": [{"label": "Expected Shortfall ($)", "data": risk_metrics_df["expected_shortfall"].round(0).tolist()}],
         "story": STORY_LOSS_DIST},
    ],
    insights=INSIGHTS,
    data_table={"title": "Risk Metrics by Confidence Level", "columns": list(risk_metrics_df.columns),
                "rows": risk_metrics_df.values.tolist()},
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s).")

# ---------------------------------------------------------------------------
# SECTION 14 — Save artifacts + governance stamp (idempotent)
# ---------------------------------------------------------------------------
summary = {
    "notebook": "03_economic_capital_unexpected_loss",
    "mega_project": "Mega Project 2 - Regulatory Capital & Expected Loss",
    "problem": "Problem 3 - Economic Capital & Unexpected Loss",
    "random_seed": SEED,
    "n_applicants": N_SCOPE,
    "upstream_dependency": {"source_notebook": "Mega Project 2 / Notebook 01",
                             "reused_not_recomputed": True,
                             "columns_reused": required_cols},
    "monte_carlo_config": {"n_draws_main": N_MC_DRAWS, "n_draws_reseed_check": N_MC_DRAWS_CHECK,
                            "batch_size": MC_BATCH_SIZE, "main_run_seed": SEED, "reseed_check_seed": SEED + 1,
                            "runtime_seconds_main_run": MC_RUNTIME_S, "runtime_seconds_reseed_check": CHECK_RUNTIME_S},
    "portfolio_totals": {"real_total_ead_usd": REAL_TOTAL_EAD, "real_total_expected_loss_usd": REAL_TOTAL_EL,
                          "closed_form_basel_capital_usd": CLOSED_FORM_TOTAL_CAPITAL,
                          "simulated_mean_loss_usd": float(SIMULATED_LOSSES.mean())},
    "risk_metrics_by_confidence_level": risk_metrics,
    "closed_form_cross_check": {"monte_carlo_99_9pct_economic_capital_usd": EC_999,
                                 "closed_form_basel_capital_usd": CLOSED_FORM_TOTAL_CAPITAL,
                                 "relative_difference": CLOSED_FORM_VS_MC_REL_DIFF,
                                 "documented_tolerance": CROSS_CHECK_TOLERANCE,
                                 "within_tolerance": bool(CROSS_CHECK_PASSES)},
    "convergence_check": convergence_detail,
    "statistical_validation": {
        "validation_checks": {name: bool(ok) for name, ok in validation_checks},
        "failed_validation_checks": _failed_validation_checks,
        "deployment_verdict": ANALYSIS_VERDICT,
        "note": "This notebook validates a Monte Carlo simulation, not a classifier against real TARGET -- "
                "'statistical robustness' here means independent-reseed convergence plus closed-form "
                "cross-check, not chi-square/Cramer's V (see module docstring).",
    },
    "integrity_checks": {n: bool(ok) for n, ok in checks},
    "reporting_artifacts": ["notebook_03_report.docx", "notebook_03_workbook.xlsx", "notebook_03_dashboard.html"]
                           + [f"{stem}.csv" for stem in csv_paths],
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_03_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"[DONE] Mega Project 2 / Notebook 03 complete in {summary['runtime_seconds']}s. "
      f"Real 99.9% Economic Capital: ${EC_999:,.0f} ({CLOSED_FORM_VS_MC_REL_DIFF:.2%} vs. closed-form capital). "
      f"Statistical robustness verdict: {ANALYSIS_VERDICT}.")
